# ETL The Look: Sales and Stock Data Warehouse

**Author:** Narda Parama Agung Noviantoro

This project builds a PostgreSQL data warehouse that joins sales and stock data from The Look, an e-commerce company, so the company can see which products sell fast and which stay unsold in its distribution centers. It was originally built as Graded Challenge 6 of the Hacktiv8 Data Analytics Bootcamp.

**ETL process:**
- **Extract:** pull 2025 sales and inventory data from BigQuery (`thelook_ecommerce`) into CSV files using SQL.
- **Transform:** clean the data and build 2 fact tables and 4 dimension tables with PySpark.
- **Load:** load the tables into a PostgreSQL data warehouse (galaxy schema) and answer the business problems with SQL.

**Tools:** Python, PySpark, pandas, SQL, BigQuery, PostgreSQL

# Data Modeling

## Business Problem
The Look is an e-commerce company that sells fashion products from several distribution centers. Its sales and stock data are stored separately, which creates two problems:
1. It is hard to see which products sell fastest, so it is hard to know which ones need priority restocking.
2. Some products stay in the warehouse for a long time without selling, which ties up capital in unsold stock.

**Objective:** Build a data warehouse that joins sales and stock data, so the sales and merchandising teams can find the products, categories, brands, and distribution centers with slow-moving stock, prioritize restocking for fast sellers, and add marketing for slow sellers.

**Data period:** Orders and stock received in 2025. 2026 is excluded because the year has not ended, so recently received stock would look unsold only because it is new. Sales dates can still fall in early 2026, because stock received in 2025 may be sold later.

## Business Process

The business process of sales at The Look from received stock until products are sold:
1. A product is registered and listed for sale, then placed in one of the distribution centers.
2. Product stock arrives at the distribution center and is prepared to be sold.
3. A customer registers through the platform using a certain traffic source and orders a product, then one unit from the stock is allocated for that order.
4. That stock unit is marked as sold, which allows the company to identify how long that stock stayed before being sold.
5. The product is sent from the distribution center to the customer.
6. If any problems occur or a customer is unsatisfied, some orders may be cancelled or returned by customers.
7. Stock that was not sold is kept in the distribution center and is considered slow-moving.


## Fact Table

### fact_sales (main)
Tracks each product sold to customers.

Grain: one row = one order item
- Primary key: order_item_id
- Other IDs: order_id, inventory_item_id
- Foreign key: user_id, product_id, distribution_center_id, order_date_id, shipped_date_id, delivered_date_id
- Attribute: status
- Measures: sale_price, cost, profit (new column: sale_price - cost)

### fact_inventory (supporting)
Tracks every stock unit that is ready to be sold to understand whether and how quickly that stock is sold.

Grain: one row = one stock unit that enters a distribution center.
- Primary key: inventory_item_id
- Foreign key: product_id, distribution_center_id, received_date_id, sold_date_id (null if not sold)
- Measures: cost, is_sold (new column: 1 if sold, 0 if not sold), days_in_stock (new column: total days stock units stay until sold. *it will be null if not yet sold)

## Dimension Table
- dim_product (products being sold): product_id, product_name, category, brand, department, retail_price
- dim_distribution_center (where products are stored and sent): distribution_center_id, distribution_center_name, latitude, longitude
- dim_user (who buys the products): user_id, age, gender, city, state, country, traffic_source
- dim_date (shows sales and stock dates): date_id (yyyyMMdd), full_date, year, quarter, month, month_name, day_name

## Schema Type: Galaxy Schema
This data warehouse will be using a galaxy schema because it contains two fact tables that share the same dimensions.
- dim_product, dim_distribution_center, and dim_date will be used for both fact tables.
- dim_user will only be used by fact_sales.

# Setup
Importing libraries and setting up Java for PySpark.

In [ ]:
import os, sys # imports os and sys

# configuring JAVA_HOME
os.environ.setdefault("JAVA_HOME", os.path.join(sys.prefix, "Library" if os.name == "nt" else "lib/jvm"))

Reading the database credentials from the `.env` file and creating the SparkSession with the PostgreSQL driver.

In [ ]:
from dotenv import load_dotenv # to read the .env file
from pyspark.sql import SparkSession # to create SparkSession
from pyspark.sql import functions as F # adds PySpark functions
from pyspark.sql.window import Window # to create row numbers

load_dotenv('.env') # reads the .env file

spark = SparkSession.builder \
    .appName("ETLTheLook") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.14") \
    .getOrCreate() # creates SparkSession connected to PostgreSQL

# Extract
Reading the CSV files exported from BigQuery with the queries in `sql/extract_thelook_queries.sql`.

In [3]:
def read_csv(file_name): # creates a function to read the csv files
    return spark.read.csv(f'data/{file_name}', header=True, inferSchema=True, escape='"')  # reads csv files based on the file name in the data folder

order_items = read_csv('order_items_2025.csv')  # reads 2025 order items data for fact_sales
inventory_items = read_csv('inventory_items_2025.csv')  # reads 2025 stock unit data for fact_inventory
products = read_csv('products.csv') # reads products data for dim_product
users = read_csv('users_2025.csv')  # reads 2025 users/customer data for dim_user
distribution_centers = read_csv('distribution_centers.csv') # reads distribution centers data for dim_distribution_center

# Exploration
Checking the raw data with pandas before transformation.

Loading the same CSV files into pandas dataframes, kept separate from the PySpark dataframes.

In [4]:
import pandas as pd # importing pandas library

pd_order_items = pd.read_csv('data/order_items_2025.csv')   # reads order_items data
pd_inventory_items = pd.read_csv('data/inventory_items_2025.csv')   # reads inventory_items data
pd_products = pd.read_csv('data/products.csv')  # reads products data
pd_users = pd.read_csv('data/users_2025.csv')   # reads customers/users data
pd_distribution_centers = pd.read_csv('data/distribution_centers.csv')  # reads distribution_centers data

Checking total rows and datatypes.

In [5]:
print('order_items')    # gives title
pd_order_items.info()   # shows info for order_items data

print() # create space

print('inventory_items')    # gives title
pd_inventory_items.info()   # shows info for inventory_items data

print() # create space

print('products')   # gives title
pd_products.info()  # shows info for products data

print() # create space

print('users')  # gives title
pd_users.info() # shows info for customers/users data

print() # create space

print('distribution_centers')   #    gives title
pd_distribution_centers.info()  # shows info for distribution_centers data

order_items
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41019 entries, 0 to 41018
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 41019 non-null  int64  
 1   order_id           41019 non-null  int64  
 2   user_id            41019 non-null  int64  
 3   product_id         41019 non-null  int64  
 4   inventory_item_id  41019 non-null  int64  
 5   status             41019 non-null  object 
 6   created_at         41019 non-null  object 
 7   shipped_at         26845 non-null  object 
 8   delivered_at       14362 non-null  object 
 9   sale_price         41019 non-null  float64
dtypes: float64(1), int64(5), object(4)
memory usage: 3.1+ MB

inventory_items
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87625 entries, 0 to 87624
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  --

Date columns (created_at, shipped_at, delivered_at, sold_at) are stored as text, so they need to be converted to date type.

Checking if there are any null values.

In [6]:
print('order_items')    # gives title
print(pd_order_items.isnull().sum())    # checks null values in order_items data

print() # create space

print('inventory_items')    # gives title
print(pd_inventory_items.isnull().sum())    # checks null values in inventory_items data

print() # create space

print('products')   # gives title
print(pd_products.isnull().sum())   # checks null values in products data

print() # create space

print('users')  # gives title
print(pd_users.isnull().sum())    # checks null values in customers/users data

print() # creates space

print('distribution_centers')   #    gives title
print(pd_distribution_centers.isnull().sum())   # checks null values in distribution_centers data

order_items
id                       0
order_id                 0
user_id                  0
product_id               0
inventory_item_id        0
status                   0
created_at               0
shipped_at           14174
delivered_at         26657
sale_price               0
dtype: int64

inventory_items
id                                    0
product_id                            0
created_at                            0
sold_at                           45075
cost                                  0
product_distribution_center_id        0
dtype: int64

products
id                         0
name                       2
category                   0
brand                     24
department                 0
retail_price               0
cost                       0
distribution_center_id     0
dtype: int64

users
id                  0
age                 0
gender              0
city              244
state               0
country             0
traffic_source      0
dtype: int64

distr

shipped_at, delivered_at, and sold_at have nulls because some orders are not shipped or delivered yet and some stock is not sold yet, so these stay null. name (2), brand (24), and city (244) have a few nulls, which are filled with 'Unknown'.

Checking total duplicate rows and ID duplicates.

In [7]:
print('order_items', pd_order_items.duplicated().sum(), pd_order_items['id'].duplicated().sum())    # checks duplicate rows and id duplicates in order_items data
print() # create space

print('inventory_items', pd_inventory_items.duplicated().sum(), pd_inventory_items['id'].duplicated().sum())    # checks duplicate rows and id duplicates in inventory_items data
print() # create space

print('products', pd_products.duplicated().sum(), pd_products['id'].duplicated().sum())    # checks duplicate rows and id duplicates in products data
print() # create space

print('users', pd_users.duplicated().sum(), pd_users['id'].duplicated().sum())    # checks duplicate rows and id duplicates in customers/users data
print() # create space

print('distribution_centers', pd_distribution_centers.duplicated().sum(), pd_distribution_centers['id'].duplicated().sum())    # checks duplicate rows and id duplicates in distribution_centers data

order_items 0 0

inventory_items 0 0

products 0 0

users 0 0

distribution_centers 0 0


There are no duplicate rows or duplicate IDs.

# Transformation
Data cleaning and creating the fact and dimension tables using PySpark.

## Cleaning

Cleaning products: rename columns, fill missing name and brand with 'Unknown', round cost to 2 decimals, and keep one row per product_id.

In [8]:
products_clean = (products  # raw data
    .drop_duplicates()  # drops duplicate rows
    .withColumnRenamed('id', 'product_id')  # change column name from 'id' to 'product_id'
    .withColumnRenamed('name', 'product_name') # change column name from 'name' to 'product_name'
    .fillna({'product_name': 'Unknown', 'brand': 'Unknown'})    # fill null values with 'Unknown'
    .withColumn('cost', F.round('cost', 2)) # rounds cost to two decimal places
    .drop_duplicates(subset=['product_id']))    # ensuring that there is only one product id per row

products_clean.show(5)  # shows the first 5 rows

+----------+--------------------+-----------+------------------+----------+------------+-----+----------------------+
|product_id|        product_name|   category|             brand|department|retail_price| cost|distribution_center_id|
+----------+--------------------+-----------+------------------+----------+------------+-----+----------------------+
|         1|Seven7 Women's Lo...|Tops & Tees|            Seven7|     Women|        49.0|27.05|                     1|
|         3|Calvin Klein Jean...|Tops & Tees|Calvin Klein Jeans|     Women|        69.5| 41.0|                     8|
|         5|Anne Klein Women'...|Tops & Tees|        Anne Klein|     Women|        94.0|52.36|                     2|
|         6|Wilt Women's Colo...|Tops & Tees|              Wilt|     Women|       132.0|78.94|                     6|
|        12|Jones New York Wo...|Tops & Tees|    Jones New York|     Women|        99.0|50.79|                     3|
+----------+--------------------+-----------+-----------

Cleaning data distribution_center by changing column names.

In [9]:
distribution_centers_clean = (distribution_centers  # raw data
    .drop_duplicates()  # drops duplicate rows
    .withColumnRenamed('id', 'distribution_center_id') # change column name from 'id' to 'distribution_centers_id'
    .withColumnRenamed('name', 'distribution_center_name'))    # change column name 'name' to 'distribution_centers_name'

distribution_centers_clean.show(5)  # shows the first 5 rows

+----------------------+------------------------+--------+---------+
|distribution_center_id|distribution_center_name|latitude|longitude|
+----------------------+------------------------+--------+---------+
|                     9|           Charleston SC| 32.7833| -79.9333|
|                     4|          Los Angeles CA|   34.05|  -118.25|
|                     5|          New Orleans LA|   29.95| -90.0667|
|                     1|              Memphis TN| 35.1174| -89.9711|
|                     6|    Port Authority of...|  40.634| -73.7834|
+----------------------+------------------------+--------+---------+
only showing top 5 rows


Cleaning customers/users data by changing column names and filling in null values in city.

In [10]:
users_clean = (users    # raw data
    .drop_duplicates()  # drops duplicate rows
    .withColumnRenamed('id', 'user_id') # change column name from 'id' to 'user_id'
    .fillna({'city': 'Unknown'})    # fills in null values with 'Unknown'
    .dropDuplicates(subset=['user_id']))    # ensuring that there is only user id per row

users_clean.show(5) # shows the first 5 rows

+-------+---+------+------------+--------------+-------------+--------------+
|user_id|age|gender|        city|         state|      country|traffic_source|
+-------+---+------+------------+--------------+-------------+--------------+
|      1| 18|     F|  Auburndale|       Florida|United States|        Search|
|      3| 25|     F|      Lavras|  Minas Gerais|       Brasil|         Email|
|      6| 52|     M|Santa Isabel|     São Paulo|       Brasil|       Display|
|     11| 16|     M|   Greenwood|South Carolina|United States|        Search|
|     18| 36|     M|   São Paulo|     São Paulo|       Brasil|       Organic|
+-------+---+------+------------+--------------+-------------+--------------+
only showing top 5 rows


Cleaning order items: rename id to order_item_id and convert the timestamp columns to date type.

In [11]:
order_items_clean = (order_items    # raw data
    .drop_duplicates()  # drops duplicate rows
    .withColumnRenamed('id', 'order_item_id')   # change column name from 'id' to 'order_item_id'
    .withColumn('order_date', F.to_date('created_at'))  # change order_date into date format
    .withColumn('shipped_date', F.to_date('shipped_at'))    # change shipped_date into date format
    .withColumn('delivered_date', F.to_date('delivered_at'))    # change delivered_date into date format
    .drop('created_at', 'shipped_at', 'delivered_at'))   # deletes old date columns

order_items_clean.show(5)   # shows the first 5 rows

+-------------+--------+-------+----------+-----------------+--------+------------------+----------+------------+--------------+
|order_item_id|order_id|user_id|product_id|inventory_item_id|  status|        sale_price|order_date|shipped_date|delivered_date|
+-------------+--------+-------+----------+-----------------+--------+------------------+----------+------------+--------------+
|        80840|   55667|  44712|     14220|           218129|Returned| 8.770000457763672|2025-08-26|  2025-08-25|    2025-08-26|
|       130085|   89624|  71945|       576|           351226|Complete|  8.90999984741211|2025-06-27|  2025-06-28|    2025-06-30|
|        55125|   37979|  30494|     25343|           148621|Complete|              10.0|2025-12-07|  2025-12-10|    2025-12-12|
|        31835|   22012|  17595|      4077|            85841|Complete| 10.81999969482422|2025-09-13|  2025-09-10|    2025-09-14|
|        10173|    7028|   5539|       221|            27449|Returned|11.010000228881836|2025-11-

Cleaning inventory items: rename columns, convert the timestamp columns to date type, and round cost to 2 decimals.

In [12]:
inventory_items_clean = (inventory_items    # raw data
    .drop_duplicates() # drops duplicate rows
    .withColumnRenamed('id', 'inventory_item_id')   # change column name from 'id' to 'inventory_item_id'
    .withColumnRenamed('product_distribution_center_id', 'distribution_center_id')  # change column name from 'product_distribution_center_id' to 'distribution_center_id'
    .withColumn('received_date', F.to_date('created_at'))   # change received_date into date format
    .withColumn('sold_date', F.to_date('sold_at'))  # change sold_date into date format
    .withColumn('cost', F.round('cost', 2)) # rounds up cost to two decimal places
    .drop('created_at', 'sold_at')) # drops old date columns

inventory_items_clean.show(5) # shows the first 5 rows

+-----------------+----------+------+----------------------+-------------+----------+
|inventory_item_id|product_id|  cost|distribution_center_id|received_date| sold_date|
+-----------------+----------+------+----------------------+-------------+----------+
|           465637|     28830|   7.6|                     3|   2025-05-28|2025-06-13|
|           233029|     28626| 29.66|                     1|   2025-06-04|2025-07-05|
|           269157|     14329|115.41|                     8|   2025-11-23|2025-11-23|
|           446332|     28528|  4.33|                     4|   2025-05-19|2025-06-03|
|           234862|     28806| 43.32|                     8|   2025-06-04|2025-06-15|
+-----------------+----------+------+----------------------+-------------+----------+
only showing top 5 rows


Checking order and inventory items where products are not listed in the product data.

In [13]:
order_items_clean.join(products_clean, on='product_id', how='left_anti').count(), inventory_items_clean.join(products_clean, on='product_id', how='left_anti').count()  # counts the total order and inventory items where products do not match

(0, 0)

Based on the above check, all order and inventory items have matching products.

## Dimension Table

Creating the dim_product, dim_distribution_center, and dim_user tables.

In [14]:
dim_product = products_clean[['product_id', 'product_name', 'category', 'brand', 'department', 'retail_price']] # creating columns for dim_product
dim_distribution_center = distribution_centers_clean[['distribution_center_id', 'distribution_center_name', 'latitude', 'longitude']]   # creating columns for dim_distribution_center
dim_user = users_clean[['user_id', 'age', 'gender', 'city', 'state', 'country', 'traffic_source']]  # creating columns for dim_user

print('dim_product')    # creates title
dim_product.show(5) # shows the first 5 rows of dim_product
print() # creates space

print('dim_distribution_center')    # creates title
dim_distribution_center.show(5) # shows the first 5 rows of dim_distribution_center
print() # creates space

print('dim_user')   # creates title
dim_user.show(5)    # shows the first 5 rows of dim_user
print() # creates space

dim_product
+----------+--------------------+-----------+------------------+----------+------------+
|product_id|        product_name|   category|             brand|department|retail_price|
+----------+--------------------+-----------+------------------+----------+------------+
|         1|Seven7 Women's Lo...|Tops & Tees|            Seven7|     Women|        49.0|
|         3|Calvin Klein Jean...|Tops & Tees|Calvin Klein Jeans|     Women|        69.5|
|         5|Anne Klein Women'...|Tops & Tees|        Anne Klein|     Women|        94.0|
|         6|Wilt Women's Colo...|Tops & Tees|              Wilt|     Women|       132.0|
|        12|Jones New York Wo...|Tops & Tees|    Jones New York|     Women|        99.0|
+----------+--------------------+-----------+------------------+----------+------------+
only showing top 5 rows

dim_distribution_center
+----------------------+------------------------+--------+---------+
|distribution_center_id|distribution_center_name|latitude|longitude|


Building dim_date from every order, shipped, delivered, received, and sold date, with nulls and duplicates removed.

In [15]:
dates = (order_items_clean.select(F.col('order_date').alias('full_date'))  # extracts order date
    .union(order_items_clean.select(F.col('shipped_date').alias('full_date')))   # adds shipping date
    .union(order_items_clean.select(F.col('delivered_date').alias('full_date')))   # adds delivery date
    .union(inventory_items_clean.select(F.col('received_date').alias('full_date'))) # adds dates received
    .union(inventory_items_clean.select(F.col('sold_date').alias('full_date'))) # adds dates sold
    .dropna()   # drops any null values
    .distinct()) # ensures one date per row

dim_date = (dates   # the full dates data
    .withColumn('date_id', F.date_format('full_date', 'yyyyMMdd').cast('int'))  # creates date_id
    .withColumn('year', F.year('full_date'))    # creates 'year' column
    .withColumn('quarter', F.quarter('full_date'))  # creates 'quarter' column
    .withColumn('month', F.month('full_date'))    # creates 'month' column
    .withColumn('month_name', F.date_format('full_date', 'MMMM')) # creates 'month_name' column
    .withColumn('day_name', F.date_format('full_date', 'EEEE')))    # creates 'day_name' column

dim_date.show(5)    # shows the first 5 rows of dim_date

+----------+--------+----+-------+-----+----------+--------+
| full_date| date_id|year|quarter|month|month_name|day_name|
+----------+--------+----+-------+-----+----------+--------+
|2025-10-21|20251021|2025|      4|   10|   October| Tuesday|
|2025-02-16|20250216|2025|      1|    2|  February|  Sunday|
|2025-11-04|20251104|2025|      4|   11|  November| Tuesday|
|2025-06-29|20250629|2025|      2|    6|      June|  Sunday|
|2025-06-08|20250608|2025|      2|    6|      June|  Sunday|
+----------+--------+----+-------+-----+----------+--------+
only showing top 5 rows


## Fact Table

Creating fact_sales.

In [16]:
fact_sales = (order_items_clean # the cleaned order_items data
    .join(products_clean[['product_id', 'cost', 'distribution_center_id']], on='product_id', how='left')  # joins cost and distribution_centers_id on product_id
    .withColumn('sale_price', F.round('sale_price', 2))    # adds sales_price and rounds to two decimal places
    .withColumn('profit', F.round(F.col('sale_price') - F.col('cost'), 2))  # adds profit by subtracting sales_price by cost and rounding to two decimal places
    .withColumn('order_date_id', F.date_format('order_date', 'yyyyMMdd').cast('int'))  # adds order_date_id
    .withColumn('shipped_date_id', F.date_format('shipped_date', 'yyyyMMdd').cast('int'))   # adds shipped_date_id
    .withColumn('delivered_date_id', F.date_format('delivered_date', 'yyyyMMdd').cast('int'))   # adds delivered_date_id
    .select('order_item_id', 'order_id', 'inventory_item_id', 'user_id', 'product_id', 'distribution_center_id',
            'order_date_id', 'shipped_date_id', 'delivered_date_id', 'status', 'sale_price', 'cost', 'profit')) # selecting columns 

fact_sales.show(5)  # shows the first 5 rows of fact_sales

+-------------+--------+-----------------+-------+----------+----------------------+-------------+---------------+-----------------+--------+----------+----+------+
|order_item_id|order_id|inventory_item_id|user_id|product_id|distribution_center_id|order_date_id|shipped_date_id|delivered_date_id|  status|sale_price|cost|profit|
+-------------+--------+-----------------+-------+----------+----------------------+-------------+---------------+-----------------+--------+----------+----+------+
|        80840|   55667|           218129|  44712|     14220|                     6|     20250826|       20250825|         20250826|Returned|      8.77|3.45|  5.32|
|       130085|   89624|           351226|  71945|       576|                     9|     20250627|       20250628|         20250630|Complete|      8.91|5.19|  3.72|
|        55125|   37979|           148621|  30494|     25343|                     1|     20251207|       20251210|         20251212|Complete|      10.0|5.77|  4.23|
|        3

Creating fact_inventory.

In [17]:
fact_inventory = (inventory_items_clean # the cleaned inventory_items data
    .withColumn('received_date_id', F.date_format('received_date', 'yyyyMMdd').cast('int')) # creates received_date_id
    .withColumn('sold_date_id', F.date_format('sold_date', 'yyyyMMdd').cast('int')) # creates sold_date_id
    .withColumn('is_sold', F.when(F.col('sold_date').isNull(),0).otherwise(1))  # creates is_sold where if not sold (null), value will show 0, whereas if sold value will show 1
    .withColumn('days_in_stock', F.datediff('sold_date', 'received_date'))  # creates days_in_stock by calculating the difference of received_date and sold_date
    .select('inventory_item_id', 'product_id', 'distribution_center_id', 'received_date_id', 'sold_date_id',
            'cost', 'is_sold', 'days_in_stock'))    # selecting columns

fact_inventory.show(5)  # shows the first 5 rows of fact_inventory

+-----------------+----------+----------------------+----------------+------------+------+-------+-------------+
|inventory_item_id|product_id|distribution_center_id|received_date_id|sold_date_id|  cost|is_sold|days_in_stock|
+-----------------+----------+----------------------+----------------+------------+------+-------+-------------+
|           465637|     28830|                     3|        20250528|    20250613|   7.6|      1|           16|
|           233029|     28626|                     1|        20250604|    20250705| 29.66|      1|           31|
|           269157|     14329|                     8|        20251123|    20251123|115.41|      1|            0|
|           446332|     28528|                     4|        20250519|    20250603|  4.33|      1|           15|
|           234862|     28806|                     8|        20250604|    20250615| 43.32|      1|           11|
+-----------------+----------+----------------------+----------------+------------+------+------

# Load

Loading data into the created data warehouse tables.

Creating JDBC connection to PostgreSQL.

In [18]:
postgres_url = f"jdbc:postgresql://{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"   # adds PostgreSQL database address
postgres_properties = { 
    "user": os.getenv("PG_USER"),   # inputs username from .env
    "password": os.getenv("PG_PASSWORD"),    # inputs password from .env
    "driver": "org.postgresql.Driver"   # adds JDBC driver for PostgreSQL
}

Loading data to dimension tables and fact tables.

In [19]:
dim_product.write.jdbc(url=postgres_url, table="dim_product", mode="append", properties=postgres_properties)  # loads dim_product
dim_distribution_center.write.jdbc(url=postgres_url, table="dim_distribution_center", mode="append", properties=postgres_properties)  # loads dim_distribution_center
dim_user.write.jdbc(url=postgres_url, table="dim_user", mode="append", properties=postgres_properties)  # loads dim_user
dim_date.write.jdbc(url=postgres_url, table="dim_date", mode="append", properties=postgres_properties)  # loads dim_date
fact_sales.write.jdbc(url=postgres_url, table="fact_sales", mode="append", properties=postgres_properties)  # loads fact_sales
fact_inventory.write.jdbc(url=postgres_url, table="fact_inventory", mode="append", properties=postgres_properties)  # loads fact_inventory

26/11/10 20:45:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Analysis
Reading from the PostgreSQL data warehouse with SQL to answer the business problems.

**Analysis 1:** Average days to sell per category (sold stock only).

In [20]:
sql_query1 = """SELECT
    p.category,
    COUNT(*) AS sold_units,
    ROUND(AVG(f.days_in_stock), 1)::float AS avg_days_sell
FROM fact_inventory f
JOIN dim_product p ON f.product_id = p.product_id
WHERE f.is_sold = 1
GROUP BY p.category
ORDER BY avg_days_sell ASC"""   # finds the average days it takes for products to sell per category
query1 = f"({sql_query1}) AS subquery"  # sql_query1 is saved into query1

df1 = spark.read.jdbc(url=postgres_url, table=query1, properties=postgres_properties)   # runs the query
df1.show(50)    # shows all rows

+--------------------+----------+-------------+
|            category|sold_units|avg_days_sell|
+--------------------+----------+-------------+
|             Dresses|      1235|         28.9|
|               Pants|      1609|         29.7|
|               Socks|      1472|         29.7|
|                Plus|      1033|         29.8|
|         Tops & Tees|      2838|         29.9|
|         Accessories|      2296|         30.0|
|           Intimates|      3149|         30.0|
|     Socks & Hosiery|       849|         30.1|
|           Underwear|      1773|         30.2|
|Fashion Hoodies &...|      2784|         30.3|
|           Maternity|      1161|         30.3|
|   Outerwear & Coats|      2131|         30.4|
|      Sleep & Lounge|      2680|         30.4|
| Suits & Sport Coats|      1194|         30.4|
|               Jeans|      2987|         30.5|
|            Leggings|       740|         30.6|
|      Pants & Capris|       825|         30.6|
|              Shorts|      2694|       

Selling speed is similar across categories (28.9 to 32.5 days). Dresses sell fastest and Jumpsuits & Rompers sell slowest.

**Analysis 2:** Unsold stock and unsold percentage per distribution center.

In [21]:
sql_query2 = """SELECT
    d.distribution_center_name,
    COUNT(*) AS stock_total,
    COUNT(*) - SUM(f.is_sold) AS not_sold_stocks,
    ROUND(100.0 * (COUNT(*) - SUM(f.is_sold)) / COUNT(*), 2)::float AS unsold_percentage
FROM fact_inventory f
JOIN dim_distribution_center d ON f.distribution_center_id = d.distribution_center_id
GROUP BY d.distribution_center_name
ORDER BY unsold_percentage DESC""" # finds stocks that have not sold in each distribution center
query2 = f"({sql_query2}) AS subquery"  # sql_query 2 saved into query2

df2 = spark.read.jdbc(url=postgres_url, table=query2, properties=postgres_properties)   # runs the query
df2.show()    # shows all rows

+------------------------+-----------+---------------+-----------------+
|distribution_center_name|stock_total|not_sold_stocks|unsold_percentage|
+------------------------+-----------+---------------+-----------------+
|               Mobile AL|       8750|           4590|            52.46|
|             Savannah GA|       5731|           2980|             52.0|
|              Memphis TN|      11569|           5961|            51.53|
|          Los Angeles CA|       8425|           4340|            51.51|
|              Chicago IL|      11668|           6005|            51.47|
|              Houston TX|      11105|           5698|            51.31|
|          New Orleans LA|       6337|           3245|            51.21|
|         Philadelphia PA|       8150|           4158|            51.02|
|           Charleston SC|       8076|           4120|            51.02|
|    Port Authority of...|       7814|           3978|            50.91|
+------------------------+-----------+-------------

More than 50% of the stock received in 2025 is still unsold in every distribution center, highest in Mobile AL (52.46%) and lowest in Port Authority of New York/New Jersey (50.91%). "Unsold" means not sold in the data up to March 2026. This suggests overstocking or weak demand.

In [22]:
spark.stop()    # stops PySpark

# Conclusion
- The data warehouse (2 fact tables, 4 dimension tables) joins sales and stock, so selling speed and unsold stock can be analyzed by product, category, and distribution center.
- 45,075 of 87,625 stock units received in 2025 are unsold, and every distribution center has more than 50% unsold stock.
- Selling speed barely differs between categories (28.9 to 32.5 days).

**Limitations:** days_in_stock only exists for sold units, so it understates how long unsold units sit in stock. Unsold status depends on how far the data runs (March 2026).

# Presentation link
Google Slides: https://docs.google.com/presentation/d/1Zu3Ibv2xWMqKQh6wzns-6HijVluTHalrsG3QBMuzrBE/edit?usp=sharing